# Customer Drop-off, Conversion & Attrition Risk Analysis for an E-commerce Product

---

### Objective:
#### Analyse customer engagement, identify conversion drop-off points, and segment customers by attrition risk to support product and retention decisions.

In [ ]:
# importing the data
import pandas as pd
import numpy as np
data = pd.read_csv('../data/ecommerce_customer_churn_dataset.csv')
df = pd.DataFrame(data)
#initial inspection
df.head()

#### Inspecting the data

In [ ]:
df.info()

In [ ]:
df.shape

In [45]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,50000.0,37.775400,11.156026,18.000000,30.000000,38.000000,45.000000,75.000000
Membership_Years,50000.0,2.984009,2.059105,0.100000,1.400000,2.500000,4.000000,10.000000
Login_Frequency,50000.0,11.624660,7.810657,0.000000,6.000000,11.000000,17.000000,46.000000
Session_Duration_Avg,50000.0,27.602240,10.497234,1.000000,20.200000,26.800000,34.000000,75.600000
Pages_Per_Session,50000.0,8.717542,3.663996,1.000000,6.200000,8.400000,11.000000,24.100000
Cart_Abandonment_Rate,50000.0,57.079973,16.282723,0.000000,46.400000,58.100000,68.700000,143.743350
Wishlist_Items,50000.0,3.954520,3.274206,0.000000,1.000000,3.000000,6.000000,28.000000
Total_Purchases,50000.0,13.116560,7.005246,0.000000,8.000000,12.000000,17.000000,128.700000
Average_Order_Value,50000.0,123.117330,175.569714,26.380000,87.050000,112.970000,144.440000,9666.379178
Days_Since_Last_Purchase,47000.0,29.792872,29.695062,0.000000,9.000000,21.000000,41.000000,287.000000


### Missing Data Audit

In [ ]:
missing_summary = (
    df.isnull()
      .sum()
      .to_frame("missing_count")
      .assign(missing_percent=lambda x: x["missing_count"] / len(df) * 100)
      .sort_values("missing_percent", ascending=False)
)

missing_summary

## Data Quality and Assumption

---
Behavioural engagement metrics were imputed using the median to reduce the influence of outliers while preserving typical user behaviour. Count-based metrics were filled with zero, reflecting the absence of recorded activity. Time-based recency values were not imputed; instead, missingness was retained via an indicator flag to avoid introducing artificial recency assumptions.

In [ ]:
df['Age'] = df['Age'].fillna(value=df['Age'].median())

#### Age Data Validation

Age values outside a plausible adult customer range (below 18 or above 90) were treated as invalid and set to missing. These values likely represent data entry or synthetic generation errors rather than true customer attributes.

Invalid ages were subsequently imputed using the median to preserve the overall demographic distribution while avoiding distortion from implausible values. A flag was retained to track records affected by this adjustment.

In [ ]:
#implausible age range correction
min_age = 18
max_age = 90

# Flag implausible ages
df["age_out_of_range_flag"] = (
    (df["Age"] < min_age) | (df["Age"] > max_age)
).astype(int)

# Set implausible ages to NaN
df.loc[df["age_out_of_range_flag"] == 1, "Age"] = np.nan

# Impute with median
df["Age"] = df["Age"].fillna(df["Age"].median())

#### Invalid Purchase Count Handling
Negative values were observed in the Total_Purchases field, which are not logically possible in an e-commerce context and likely represent data generation or ingestion errors. These values were treated as invalid and set to missing.

Missing purchase counts were imputed as zero to avoid fabricating purchase activity, ensuring a conservative and business-consistent treatment. A flag was retained to track records affected by this correction.

In [ ]:
df["invalid_total_purchases_flag"] = (df["Total_Purchases"] < 0).astype(int)
df.loc[df["Total_Purchases"] < 0, "Total_Purchases"] = np.nan
df["Total_Purchases"] = df["Total_Purchases"].fillna(0)

#### Data type normalization

In [ ]:
df["Churned"] = df["Churned"].astype(int)
df["Signup_Quarter"] = df["Signup_Quarter"].astype("category")

### Column Grouping

In [ ]:
behavioral_metrics = [
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session",
    "Social_Media_Engagement_Score",
    "Email_Open_Rate",
    "Mobile_App_Usage"
]

count_metrics = [
    "Wishlist_Items",
    "Product_Reviews_Written",
    "Customer_Service_Calls"
]

time_based_metrics = [
    "Days_Since_Last_Purchase"
]

### Behavorial metrics
Behavioural engagement metrics with missing values were imputed using the median. This decision reflects the skewed nature of engagement data and ensures robust segmentation without overstating or understating user activity. Missing values are treated as unknown rather than zero activity.

In [ ]:
#Filling each of the missing values with their corresponding medians
for col in behavioral_metrics:
    median_value = df[col].median()
    df[col] = df[col].fillna(median_value)
    

In [ ]:
#validation
#df[behavioral_metrics].isnull().sum()

### Count metrics
Count metrics with missing values were filled with 0. This choice was made as missing data in these metrics highly likely implies a lack of engagement and and are useful to the analysis.

In [ ]:
for col in count_metrics:
    df[col] = df[col].fillna(0)

### Time-Based metric
This metric with the missingness needs to be preserved as any assumptions will cause distortion in the data. So a flag column will be set for missing values and the column is left untouched

In [ ]:
df["missing_last_purchase_flag"] = df["Days_Since_Last_Purchase"].isnull().astype(int)

## Funnel Construction

---
### Engagement Score
A composite engagement score was constructed using normalised login frequency, average session duration, and pages per session. A simple average was chosen to maintain interpretability and avoid arbitrary weighting. Users above the median engagement score were classified as “engaged” for funnel analysis.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

#normalising the values
engagement_cols = [
    "Login_Frequency",
    "Session_Duration_Avg",
    "Pages_Per_Session"
]

scaler = MinMaxScaler()

df[[f"{col}_norm" for col in engagement_cols]] = scaler.fit_transform(
    df[engagement_cols]
)


In [ ]:
df["engagement_score"] = df[
    [f"{col}_norm" for col in engagement_cols]
].mean(axis=1)

In [ ]:
engagement_threshold = df["engagement_score"].median()

df["engaged_flag"] = (df["engagement_score"] >= engagement_threshold).astype(int)

### Conversion and Value Features
Conversion was defined as at least one completed purchase. Revenue per customer was derived to support value-based segmentation. Cart abandonment was treated as a relative friction score and high-abandonment users were identified using percentile-based thresholds to avoid unit ambiguity.

In [ ]:
df["purchase_flag"] = (df["Total_Purchases"] >= 1).astype(int)

#### Revenue per Customer

In [ ]:
df["revenue_per_customer"] = df["Total_Purchases"] * df["Average_Order_Value"]

Revenue per customer was capped at the 99th percentile (£5,051) to limit the influence of extreme outliers likely arising from synthetic data artefacts, while preserving meaningful variation among high-value customers.

In [ ]:
revenue_cap = df["revenue_per_customer"].quantile(0.99)

df["revenue_per_customer_capped"] = np.where(
    df["revenue_per_customer"] > revenue_cap,
    revenue_cap,
    df["revenue_per_customer"]
)

#### High-value Customer Flag

In [ ]:
value_threshold = df["revenue_per_customer_capped"].median()

df["high_value_customer_flag"] = (
    df["revenue_per_customer_capped"] >= value_threshold
).astype(int)

#### High cart abandonment Flag

In [ ]:
abandonment_threshold = df["Cart_Abandonment_Rate"].quantile(0.75)

df["high_cart_abandonment_flag"] = (
    df["Cart_Abandonment_Rate"] >= abandonment_threshold
).astype(int)

### Recency and Frequency Segmentation
Customers were segmented into recency and purchase-frequency buckets to support churn risk assessment. Thresholds were chosen to reflect common retention intervention windows rather than optimise predictive accuracy.

#### Recency bucketing

In [ ]:
def recency_bucket(days):
    if pd.isnull(days):
        return "Unknown"
    elif days <= 30:
        return "Recent"
    elif days <= 90:
        return "At Risk"
    else:
        return "Dormant"

df["recency_bucket"] = df["Days_Since_Last_Purchase"].apply(recency_bucket).astype("category")

#### Purchase Frequency Bucketing

Customers were grouped into purchase-frequency segments based on cumulative purchase count. Thresholds were chosen to distinguish one-time buyers from occasional and frequent purchasers, supporting churn risk and value-based analysis.

In [ ]:
max_purchases = df["Total_Purchases"].max()

df["purchase_frequency_bucket"] = pd.cut(
    df["Total_Purchases"],
    bins=[0, 1, 5, max_purchases],
    labels=["One-time", "Occasional", "Frequent"],
    include_lowest=True
)

#### Purchase Frequency Distribution Caveat

The purchase frequency distribution is heavily skewed towards frequent purchasers, which is atypical for most e-commerce platforms. This suggests the dataset may over-represent high-activity users or reflect synthetic generation assumptions. As a result, purchase frequency is interpreted as a relative intensity signal rather than a literal customer lifecycle distribution.

In [ ]:
df["purchase_frequency_bucket"].value_counts()

#### Creating relative distribution

In [ ]:
df["purchase_frequency_quantile"] = pd.qcut(
    df["Total_Purchases"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [ ]:
df["purchase_frequency_quantile"].value_counts()

### Exploratory Proxy Funnel Construction

Due to the absence of event-level data, a customer-level proxy funnel was constructed using behavioural engagement, friction indicators, and purchase outcomes. This approach allows identification of meaningful drop-off points while remaining robust to data granularity constraints.

Note: This funnel was constructed as an initial proxy but exhibited near-total conversion between intent and purchase, indicating structural bias in the dataset.

In [ ]:
df["stage_engaged"] = df["engaged_flag"]

In [ ]:
df["stage_purchase_intent"] = (
    (df["stage_engaged"] == 1) &
    (df["high_cart_abandonment_flag"] == 0)
).astype(int)

In [ ]:
df["stage_converted"] = (
    (df["stage_purchase_intent"] == 1) &
    (df["purchase_flag"] == 1)
).astype(int)

In [ ]:
df["stage_retained"] = (
    (df["stage_converted"] == 1) &
    (df["Churned"] == 0)
).astype(int)

df["stage_churned"] = (
    (df["stage_converted"] == 1) &
    (df["Churned"] == 1)
).astype(int)

#### Funnel Summary Table

In [ ]:
funnel_counts = {
    "Engaged Users": df["stage_engaged"].sum(),
    "Purchase Intent Users": df["stage_purchase_intent"].sum(),
    "Converted Users": df["stage_converted"].sum(),
    "Retained Users": df["stage_retained"].sum()
}

funnel_df = pd.DataFrame.from_dict(
    funnel_counts,
    orient="index",
    columns=["user_count"]
)

funnel_df

In [ ]:
funnel_df["conversion_rate"] = (
    funnel_df["user_count"] /
    funnel_df["user_count"].shift(1)
)

funnel_df

In [ ]:
funnel_df["dropoff_rate"] = 1 - funnel_df["conversion_rate"]
funnel_df

### Revised Funnel: Engagement -> Conversion -> Retention

Initial conversion logic based on lifetime purchase resulted in near-total conversion among engaged users, reflecting dataset bias toward historical purchasers. Conversion was therefore refined to represent active purchasing behaviour by incorporating recency criteria, aligning the funnel with decision-making and retention use cases.

In [ ]:
df["stage_converted_strict"] = (
    (df["engaged_flag"] == 1) &
    (df["purchase_flag"] == 1) &
    (df["recency_bucket"].isin(["Recent", "At Risk"]))
).astype(int)

In [ ]:
funnel_counts_strict = {
    "Engaged Users": (df["engaged_flag"] == 1).sum(),

    "Active Converted Users": (
        df["stage_converted_strict"] == 1
    ).sum(),

    "Retained Users": (
        (df["stage_converted_strict"] == 1) &
        (df["Churned"] == 0)
    ).sum()
}

funnel_strict_df = pd.DataFrame.from_dict(
    funnel_counts_strict,
    orient="index",
    columns=["user_count"]
)

funnel_strict_df["conversion_rate"] = (
    funnel_strict_df["user_count"] /
    funnel_strict_df["user_count"].shift(1)
)

funnel_strict_df["dropoff_rate"] = 1 - funnel_strict_df["conversion_rate"]

funnel_strict_df